In [ ]:
import pandas as pd
import numpy as np

#loading the unclean / messy data
df = pd.read_csv('netflix_titles.csv')

#f: string-literal that puts the values in the message.
#.shape: used to count no. of rows and columns in a dataset.
#[0] and [1] refer to the tuple (rows, columns)
print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 8807 rows, 12 columns


In [3]:
print("Shape:", df.shape)
print("\nColumn Names:\n", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())


Shape: (8807, 12)

Column Names:
 ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']

Data Types:
 show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

First 5 rows:
   show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United S

In [5]:
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [7]:
#=========================
# STEP 1 : HEADER CLEANING
#=========================

print(df.columns.tolist())
# .str= columns are string type
# .strip = removes spaces at the start and end of all the strings.
# str.lower() = all lowercase
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("FIX APPLIED")
print(df.columns.tolist())


['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']
FIX APPLIED
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


In [5]:
  #=============================
  #STEP 2: FIXING MISSING VALUES
  #=============================
for col in df.columns:
   df[col] = df[col].fillna('Unknown')

#filling na (nill values) with mode( most frequently occurring value) of the data of that column.
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

#filling empty dates with 'Not Available'
df['date_added'] = df['date_added'].fillna('Not Available')

df['duration'] = df['duration'].fillna('Unknown')

#checking if fixing is correct.
print("Missing values after fix:\n", df.isnull().sum())



Missing values after fix:
 show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


In [12]:
#==============================================================
#STEP 3: FIXING DURATION PROBLEM BY SPLITTING IT INTO 2 COLUMNS
#==============================================================
# Extract numbers directly using regex

# Some rows have duration in minutes in the rating column so we will fix this by moving them to duration & setting rating to Not Available.
#df.loc[df['rating'].str.contains(r'\d+ min', na=False), 'duration'] = df['rating']
#.str.contains(r'\d+ min'): This checks whether each value in the column matches a pattern using a regular expression.
#na=False: If a value in the column is missing (Not Available), this tells pandas
#.loc filters rows based on a condition.

mask = df['rating'].str.contains(r'^\d+ min$', na=False)
print(f"Rows with duration in rating column: {mask.sum()}")

# Move the misplaced value to duration
df.loc[mask, 'duration'] = df.loc[mask, 'rating']
df.loc[mask, 'rating'] = np.nan

print("Fixed! Unique ratings now:", df['rating'].unique())


Rows with duration in rating column: 0
Fixed! Unique ratings now: ['PG-13' 'TV-MA' 'PG' 'TV-14' 'TV-PG' 'TV-Y' 'TV-Y7' 'R' 'TV-G' 'G'
 'NC-17' nan 'NR' 'Unknown' 'TV-Y7-FV' 'UR']


In [ ]:
#==============================================
#STEP 4:  Split Duration into Two Clean Columns
#==============================================

def parse_duration(row):
    d = str(row['duration'])
    if 'min' in d.lower():
        nums = re.findall(r'\d+', d)
        return pd.Series([int(nums[0]) if nums else np.nan, np.nan])
    elif 'season' in d.lower():
        nums = re.findall(r'\d+', d)
        return pd.Series([np.nan, int(nums[0]) if nums else np.nan])
    else:
        return pd.Series([np.nan, np.nan])

df[['duration_minutes', 'duration_seasons']] = df.apply(parse_duration, axis=1)

# Verify
print(df[['type', 'duration', 'duration_minutes', 'duration_seasons']].head(6))

In [13]:
!pip install ftfy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00


In [14]:
import ftfy

In [16]:
#==================================================
#STEP 5:  Fixing Special Characters in Descriptions
#==================================================
df['description'] = df['description'].apply(lambda x: ftfy.fix_text(str(x)).strip())

# Verify a few
print(df['description'].head(5).tolist())


['As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.', 'After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth.', 'To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war.', 'Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice Center in New Orleans on this gritty reality series.', "In a city of coaching centers known to train India's finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life."]


In [17]:
#==================================================
#STEP 6
#==================================================
# Fix date_added to proper datetime
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before - len(df)}")

# Strip whitespace from all text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Standardize 'type' column
df['type'] = df['type'].str.title()

print("\nFinal shape:", df.shape)
print("\nFinal missing values:\n", df.isnull().sum())

Duplicates removed: 0

Final shape: (8807, 14)

Final missing values:
 show_id                0
type                   0
title                  0
director               0
cast                   0
country                0
date_added            10
release_year           0
rating                 3
duration               0
listed_in              0
description            0
duration_minutes    2679
duration_seasons    6131
dtype: int64


In [19]:
#10 dates, 3 ratings still missing
# Since date_added is already datetime, just fill the NaT values directly
df['date_added'] = df['date_added'].fillna('Not Available')

# Fix rating
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

# Verify
print(df.isnull().sum())


show_id                0
type                   0
title                  0
director               0
cast                   0
country                0
date_added             0
release_year           0
rating                 0
duration               0
listed_in              0
description            0
duration_minutes    2679
duration_seasons    6131
dtype: int64


In [21]:
df.to_csv('netflix_titles_cleaned.csv', index=False)
print("Saved as netflix_titles_cleaned.csv")

Saved as netflix_titles_cleaned.csv
